In [0]:
from pyspark.sql.functions import explode, col

# Read the JSON files
df_stns_raw = spark.read.option("multiline", "true").json("/Volumes/railway_analytics/bronze/raw_landing/stations.json")

# Convert the nested array into rows
df_stations = df_stns_raw.select(explode(col("stations")).alias("stn")).select("stn.stnCode", "stn.stnName", "stn.stnCity")

df_stations.write.mode("overwrite").saveAsTable("railway_analytics.silver.dim_stations")


In [0]:
from pyspark.sql.functions import explode, col

# 1. Load the nested Train JSON
df_trains_json = spark.read.option("multiline", "true").json("/Volumes/railway_analytics/bronze/raw_landing/trains.json")

# 2. Extract Static Train Info (dim_trains)
df_dim_trains = df_trains_json.select(
    col("trainNumber").alias("train_no"),
    "trainName",
    "route"
)
df_dim_trains.write.mode("overwrite").saveAsTable("railway_analytics.silver.dim_trains")

# 3. Flatten the Route Info 
df_schedules = df_trains_json.select(
    col("trainNumber").alias("train_no"),
    explode(col("trainRoute")).alias("route_stop")
).select(
    "train_no",
    col("route_stop.sno").cast("int").alias("sno"),              # Added alias
    col("route_stop.stationName").alias("station_name_raw"),     # Added alias
    col("route_stop.arrives").alias("arrival_time"),            # Added alias
    col("route_stop.departs").alias("departure_time"),          # Added alias
    col("route_stop.distance").alias("distance_text"),          # Added alias
    col("route_stop.day").cast("int").alias("journey_day")      # Added alias
)

# This will now write successfully to the Silver layer
df_schedules.write.mode("overwrite").saveAsTable("railway_analytics.silver.fact_train_schedules")

print("Table 'fact_train_schedules' created successfully with clean column names.")

Table 'fact_train_schedules' created successfully with clean column names.


In [0]:
from pyspark.sql.functions import col

# Load the schedule table we just flattened
df_schedules = spark.table("railway_analytics.silver.fact_train_schedules")

# Cast train_no to Integer to remove leading zeros (e.g., '00961' -> 961)
df_schedules_fixed = df_schedules.withColumn("train_no", col("train_no").cast("int"))

# Overwrite with the fixed schema
df_schedules_fixed.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("railway_analytics.silver.fact_train_schedules")